In [1]:
import pickle
import numpy as np
from structure.node import Node
from handlers.network_handler import NetworkHandler
import os

file = open('../inputs/wilson/random_weekeday_2.pkl', 'rb')
payload_wilson_initial = pickle.load(file)
file.close()

In [2]:
from handlers.network_handler import NetworkHandler
from structure.node import Node

def get_stats(payload,manifest):
    feasible = True
    stats = {}
    stats["vmt"] = 0
    stats["pmt"] = 0
    stats["serviced"] = 0
    stats["wait_time"] = []
    stats["detour"] = []

    NetworkHandler.init(True, "http://127.0.0.1:50000/")
    request_stops = {}
    for driver_run in manifest:
        # print(driver_run["state"]["run_id"])
        load = 0
        current_node = Node(payload["depot"]["pt"]["lat"],payload["depot"]["pt"]["lon"])
        current_time = driver_run["state"]["start_time"]
        for stop in driver_run["manifest"]:
            booking_id = stop["booking_id"]
            if booking_id not in request_stops:
                request_stops[booking_id] = {}
            action = stop["action"]
            served_time = stop["scheduled_time"]
            next_node = Node(stop["loc"]["lat"],stop["loc"]["lon"])
            duration = NetworkHandler.travel_time(current_node,next_node)
            stats["vmt"] += duration
            current_time += duration
            if current_time > served_time+0.5:
                feasible = False
                print("Error: Scheduled time is impossible ", current_time-served_time)
                print("Current time: ",current_time)
                print("Scheduled time: ",served_time)
                print(stop)
            if current_time < served_time:
                current_time = served_time
            
            if served_time < stop["time_window_start"]:
                feasible = False
                print("Error: Served before window start")
            if served_time > stop["time_window_end"]:
                feasible = False
                print("Error: Served after window end")
            if action == "pickup":
                load += stop["am"]
                current_time += 180
                if "pick_up" in request_stops[booking_id]:
                    print("Error: Pick up already exists")
                request_stops[booking_id]["pick_up"] = stop
            else:
                current_time += 60
                load -= stop["am"]
                if "drop_off" in request_stops[booking_id]:
                    feasible = False
                    print("Error: Drop off already exists")
                if "pick_up" not in request_stops[booking_id]:
                    feasible = False
                    print("Error: Drop off before pick up")
                request_stops[booking_id]["drop_off"] = stop
                stats["serviced"] += 1
            if load > driver_run["state"]["am_capacity"]:
                feasible = False
                print("Error: Over capacity")
            current_node = next_node

    for served in request_stops:
        if "drop_off" not in request_stops[served]:
            feasible = False
            print("Error: Request not dropped off")
        origin = Node(request_stops[served]["pick_up"]["loc"]["lat"],request_stops[served]["pick_up"]["loc"]["lon"])
        destination = Node(request_stops[served]["drop_off"]["loc"]["lat"],request_stops[served]["drop_off"]["loc"]["lon"])
        travel_time = NetworkHandler.travel_time(origin,destination)
        stats["pmt"] += travel_time
        stats["wait_time"].append(request_stops[served]["pick_up"]["scheduled_time"]-request_stops[served]["pick_up"]["time_window_start"])
        stats["detour"].append(request_stops[served]["drop_off"]["scheduled_time"]-request_stops[served]["pick_up"]["scheduled_time"]-travel_time)
    return feasible, stats

# feasible, stats = get_stats(payload_wilson_initial,driver_runs)

In [4]:
from datetime import datetime, timedelta

# Convert seconds to timestamp
timestamp = (datetime.min + timedelta(seconds=28839)).time()
timestamp

datetime.time(8, 0, 39)

In [3]:
from online_rtv_solver import OnlineRTVSolver

payload = payload_wilson_initial
interval = 10*60 # 10 minutes
step_size = 10*60 # 10 minutes

# Initialize the RTV solver with the URL of the OSRM server
online_rtv_solver = OnlineRTVSolver("http://127.0.0.1:50000/")


# get the start_time
start_time = 24*3600
end_time = 0

for request in payload["requests"]:
    if request["pickup_time_window_start"] < start_time:
        start_time = request["pickup_time_window_start"]
    if request["dropoff_time_window_end"] > end_time:
        end_time = request["dropoff_time_window_end"]

current_time = max(0,start_time - interval)
driver_runs = payload["driver_runs"]

while current_time < end_time:
    # select the requests that are to be considered in the current interval
    selected_requests = {}
    for request in payload["requests"]:
        if request["pickup_time_window_start"] < current_time + interval and request["pickup_time_window_start"] >= current_time:
            selected_requests[request["booking_id"]] = request

    for dr in driver_runs:
        for stop in dr["manifest"]:
            if stop["booking_id"] in selected_requests:
                del selected_requests[stop["booking_id"]]
    
    selected_requests = list(selected_requests.values())


    # create a new payload with the selected requests
    new_payload = {
        "depot": payload_wilson_initial["depot"],
        "requests": selected_requests,
        "driver_runs": driver_runs}

    print("Current time: ", current_time)
    print("Number of selected requests: ", len(selected_requests))
    # solve the RTV problem
    if len(selected_requests) == 0:
        new_driver_runs = driver_runs
    else:
        # break             
        new_driver_runs,_ = online_rtv_solver.solve_rtv_fast(new_payload)
        # feasibility, stats = get_stats(payload_wilson_initial,new_driver_runs)
        # if not feasibility:
        #     print("Infeasible solution")
        #     break
    current_time += step_size

    # simulate the driver runs
    simulated_driver_runs = online_rtv_solver.simulate_manifest(current_time,new_driver_runs,intermediate_location=False)
    driver_runs = simulated_driver_runs



Current time:  19443
Number of selected requests:  0
Current time:  20043
Number of selected requests:  4
Time to initialize pool:  0.011322021484375
Time to initialize pool:  0.0071218013763427734
Time to initialize pool:  0.00718998908996582
Time to initialize pool:  0.0081329345703125
Time to initialize pool:  0.015077590942382812
Time to initialize pool:  0.00816488265991211
Time to initialize pool:  0.008015871047973633
Time to initialize pool:  0.007831573486328125
Time to initialize pool:  0.01139688491821289
Time to initialize pool:  0.011183023452758789
Time to initialize pool:  0.008261680603027344
Time to initialize pool:  0.007639884948730469
Time to initialize pool:  0.012353181838989258
Time to initialize pool:  0.013725042343139648
Time to initialize pool:  0.013070106506347656
Time to initialize pool:  0.008378267288208008
Current time:  20643
Number of selected requests:  4
Time to initialize pool:  0.015118837356567383
Time to initialize pool:  0.017781734466552734
Ti

In [4]:
feasibility, stats = get_stats(payload_wilson_initial,driver_runs)
stats["vmt"]/stats["pmt"], stats["serviced"]/len(payload_wilson_initial["requests"]), np.mean(stats["wait_time"]), np.mean(stats["detour"])

(1.4857236643565446, 0.9695121951219512, 639.6597484276721, 526.6333333333326)

In [6]:
a = 0.011843442916870117+0.009526491165161133+0.008256196975708008+0.009224891662597656+0.013826131820678711+0.008852958679199219+0.008933782577514648+0.00818943977355957+0.014590740203857422+0.013125181198120117+0.007413625717163086+0.00695490837097168+0.01298069953918457+0.015194892883300781+0.012184619903564453+0.0077648162841796875+0.015604734420776367+0.022664546966552734+0.007609367370605469+0.009307384490966797+0.016066312789916992+0.0226290225982666+0.010832071304321289+0.017281532287597656+0.027846097946166992+0.022750377655029297+0.009066343307495117+0.013783931732177734+0.024275779724121094+0.023211002349853516+0.009455442428588867+0.02335071563720703+0.008264780044555664+0.009010791778564453+0.008698463439941406+0.008620738983154297+0.010417461395263672+0.010770797729492188+0.007429838180541992+0.00834512710571289+0.02309894561767578+0.008924722671508789+0.008496284484863281+0.008426666259765625+0.030062198638916016+0.008638143539428711+0.007973432540893555+0.008946657180786133+0.03739213943481445+0.008067131042480469+0.008751869201660156+0.016702890396118164+0.03858494758605957+0.0062525272369384766+0.009692907333374023+0.011936664581298828+0.0395808219909668+0.0071887969970703125+0.008763313293457031+0.021172046661376953+0.040674686431884766+0.006083011627197266+0.010392904281616211+0.03193926811218262+0.0375518798828125+0.005735158920288086+0.012468576431274414+0.03371024131774902+0.03535103797912598+0.008575439453125+0.026669979095458984+0.03637814521789551+0.02768421173095703+0.007727384567260742+0.032662153244018555+0.02809906005859375+0.029987335205078125+0.015036582946777344+0.032303571701049805+0.02700662612915039+0.03769397735595703+0.025439023971557617+0.029484033584594727+0.024073362350463867+0.021100282669067383+0.020375728607177734+0.027359962463378906+0.016017675399780273+0.028515338897705078+0.03798961639404297+0.026283979415893555+0.016420364379882812+0.02324533462524414+0.03317093849182129+0.02523517608642578+0.024560928344726562+0.015500068664550781+0.026328563690185547+0.020462751388549805+0.031392574310302734+0.016261816024780273+0.025686264038085938+0.031095504760742188+0.03584694862365723+0.0272524356842041+0.030787944793701172+0.02566242218017578+0.02892899513244629+0.03762245178222656+0.0279691219329834+0.028642654418945312+0.027774810791015625+0.03688955307006836+0.03029346466064453+0.031426191329956055+0.025995969772338867+0.02503824234008789+0.02263355255126953+0.022847652435302734+0.02613043785095215+0.028529882431030273+0.032112836837768555+0.012142419815063477+0.024842262268066406+0.027458667755126953+0.034438133239746094+0.015991687774658203+0.03591513633728027+0.027074575424194336+0.026511430740356445+0.010118484497070312+0.028629779815673828+0.02258753776550293+0.027045011520385742+0.008648872375488281+0.04221940040588379+0.044078826904296875+0.02426600456237793+0.006982088088989258+0.040482282638549805+0.04424595832824707+0.029675960540771484+0.01665973663330078+0.03768491744995117+0.04062056541442871+0.031861066818237305+0.022875070571899414+0.04029393196105957+0.03371024131774902+0.04194307327270508+0.02340984344482422+0.03250765800476074+0.05492734909057617+0.051407575607299805+0.02607440948486328+0.043692827224731445+0.026653289794921875+0.028658390045166016+0.015157699584960938+0.042482614517211914+0.02373814582824707+0.025303363800048828+0.027558088302612305+0.03021407127380371+0.014306783676147461+0.010785341262817383+0.01756763458251953+0.01501321792602539+0.010997533798217773+0.01568603515625+0.0217742919921875+0.031557559967041016+0.012319803237915039+0.016624927520751953+0.025627851486206055+0.038956642150878906+0.007693767547607422+0.0075337886810302734+0.013387680053710938+0.02358102798461914+0.0068874359130859375+0.008210420608520508+0.012375116348266602+0.0313873291015625+0.007158756256103516+0.007613420486450195+0.006421804428100586+0.026415586471557617+0.014427661895751953+0.008082151412963867+0.007790327072143555+0.0277097225189209+0.013315916061401367+0.0078105926513671875+0.018556594848632812+0.025107860565185547+0.021692276000976562+0.0072803497314453125+0.011436223983764648+0.024517297744750977+0.018369674682617188+0.007704019546508789+0.02163386344909668+0.013626575469970703+0.01810932159423828+0.011373519897460938+0.021563291549682617+0.012477874755859375+0.016803741455078125+0.01492929458618164+0.02296614646911621+0.02154254913330078+0.005140542984008789+0.014200448989868164+0.02129220962524414+0.01591658592224121+0.007564067840576172+0.01710224151611328+0.017551660537719727+0.025709152221679688+0.014148235321044922+0.017530202865600586+0.016330480575561523+0.025482654571533203+0.0162811279296875+0.0237729549407959+0.01804518699645996+0.028591156005859375+0.018510818481445312+0.021398067474365234+0.012044429779052734+0.014291524887084961+0.019190073013305664+0.018085718154907227+0.01513671875+0.023125171661376953+0.02190685272216797+0.023775100708007812+0.014975786209106445+0.01957392692565918+0.027953624725341797+0.022324562072753906+0.013070821762084961+0.019668102264404297+0.02922987937927246+0.016738176345825195+0.012773275375366211+0.01316523551940918+0.03025984764099121+0.011413335800170898+0.015581130981445312+0.019374847412109375+0.025115966796875+0.020686626434326172+0.018822431564331055+0.017685890197753906+0.030742406845092773+0.02699422836303711+0.02979564666748047+0.019117116928100586+0.0053865909576416016+0.007025003433227539+0.007338047027587891+0.007042884826660156+0.018909692764282227+0.007628679275512695+0.00847625732421875+0.007581233978271484+0.02045893669128418+0.007669925689697266+0.008107900619506836+0.006813526153564453+0.02886176109313965+0.0069217681884765625+0.013589143753051758+0.00603175163269043+0.030443429946899414+0.0057260990142822266+0.008385181427001953+0.0046541690826416016+0.02957940101623535+0.004856586456298828+0.014590024948120117+0.005992889404296875+0.026838064193725586+0.012337923049926758+0.011801958084106445+0.006718635559082031+0.022380828857421875+0.016292333602905273+0.01181173324584961+0.007913589477539062+0.018071889877319336+0.015519857406616211+0.02051830291748047+0.008046627044677734+0.008242607116699219+0.011621475219726562+0.008223533630371094+0.008255720138549805+0.006199359893798828+0.008544206619262695+0.010015726089477539+0.009185552597045898+0.01901698112487793+0.007648468017578125+0.007993698120117188+0.007792949676513672+0.01824355125427246+0.008213520050048828+0.00730443000793457+0.0073146820068359375+0.01906418800354004+0.008517742156982422+0.0183866024017334+0.005323648452758789+0.015956401824951172+0.007710933685302734+0.019002914428710938+0.008056163787841797+0.02464604377746582+0.007778167724609375+0.0118255615234375+0.012305021286010742+0.02134227752685547+0.007640838623046875+0.007008552551269531+0.017195463180541992+0.021728515625+0.007676362991333008+0.010624408721923828+0.025083541870117188+0.024001359939575195+0.00835275650024414+0.016946077346801758+0.022082805633544922+0.012775897979736328+0.009659767150878906+0.01785135269165039+0.018679380416870117+0.016066551208496094+0.009604930877685547+0.017101287841796875+0.017080307006835938+0.01164555549621582+0.003779888153076172+0.02765488624572754+0.02691054344177246+0.004974365234375+0.009029626846313477+0.02067089080810547+0.011998176574707031+0.004919767379760742+0.006410360336303711+0.025534391403198242+0.015507936477661133+0.017362117767333984+0.007508039474487305+0.03050971031188965+0.017336368560791016+0.02572321891784668+0.007428407669067383+0.03221583366394043+0.016218900680541992+0.0261685848236084+0.0060541629791259766+0.021351099014282227+0.011690616607666016+0.024600982666015625+0.008193492889404297+0.022250652313232422+0.01958775520324707+0.025272369384765625+0.0076100826263427734+0.02395009994506836+0.030270099639892578+0.02598118782043457+0.005330324172973633+0.023207902908325195+0.03828907012939453+0.026143550872802734+0.008462905883789062+0.016696929931640625+0.03644990921020508+0.02427530288696289+0.013645410537719727+0.022162437438964844+0.03795576095581055+0.03146052360534668+0.023776531219482422+0.018333911895751953+0.03515219688415527+0.015505790710449219+0.02161383628845215+0.019254446029663086+0.03442978858947754+0.021970033645629883+0.027617692947387695+0.01314091682434082+0.03353714942932129+0.02155447006225586+0.03386211395263672+0.01858043670654297+0.03492569923400879+0.020766735076904297+0.028835773468017578+0.03289151191711426+0.03005838394165039+0.020307302474975586+0.02826237678527832+0.020866870880126953+0.02766585350036621+0.021408796310424805+0.026842355728149414+0.029132843017578125+0.030511140823364258+0.020084619522094727+0.027242422103881836+0.029890775680541992+0.0235903263092041+0.022997617721557617+0.03414440155029297+0.030281543731689453+0.02933812141418457+0.02251720428466797+0.033338069915771484+0.029332399368286133+0.028876543045043945+0.02451610565185547+0.03392481803894043+0.029286861419677734+0.035552978515625+0.02758312225341797+0.027714967727661133+0.013908624649047852+0.03417325019836426+0.014621734619140625+0.007978439331054688+0.01615285873413086+0.012157917022705078+0.008776187896728516+0.007205486297607422+0.008417129516601562+0.007844924926757812+0.008108139038085938+0.0081329345703125+0.01264500617980957+0.00889134407043457+0.005612611770629883+0.0071184635162353516+0.021083831787109375+0.00793313980102539+0.0072116851806640625+0.006747007369995117+0.022179603576660156+0.011774301528930664+0.005415201187133789+0.003818511962890625+0.017353534698486328+0.01314234733581543+0.007323265075683594+0.0074863433837890625+0.016335725784301758+0.02218151092529297+0.0051724910736083984+0.0067386627197265625+0.011464834213256836+0.03032088279724121+0.010112762451171875+0.00721287727355957+0.013267278671264648+0.024960756301879883+0.022808313369750977+0.007405519485473633+0.014971733093261719+0.025151491165161133+0.019284725189208984+0.007932901382446289+0.004731893539428711+0.011613845825195312+0.019514083862304688+0.008025169372558594+0.007013559341430664+0.0053060054779052734+0.005749940872192383+0.0069353580474853516+0.007489919662475586+0.008202552795410156+0.012618780136108398+0.005776405334472656+0.007643461227416992+0.006572246551513672+0.02203059196472168+0.00684046745300293+0.007998466491699219+0.008032798767089844+0.022446632385253906+0.006356716156005859+0.006345272064208984+0.007256507873535156+0.01757025718688965+0.007659196853637695+0.0060994625091552734+0.0047986507415771484+0.012456178665161133+0.00787663459777832+0.006429910659790039+0.015114068984985352+0.007715702056884766+0.005292177200317383+0.00569915771484375+0.007100820541381836+0.009132862091064453+0.013586044311523438+0.0071125030517578125+0.00805521011352539+0.006406307220458984+0.016752243041992188+0.00834798812866211+0.004500389099121094+0.006894588470458984+0.0192258358001709+0.007230043411254883+0.01191091537475586+0.008101701736450195+0.01949453353881836+0.0058269500732421875+0.01497960090637207+0.0066258907318115234+0.00402379035949707+0.006705284118652344+0.005149126052856445+0.006199836730957031+0.006935596466064453+0.006988048553466797+0.018216371536254883+0.005343914031982422+0.006848812103271484+0.0074138641357421875+0.009528636932373047+0.009146690368652344+0.0065038204193115234+0.008011341094970703+0.019736528396606445+0.007431507110595703+0.005850553512573242+0.006379604339599609+0.024972200393676758+0.006638288497924805+0.007169246673583984+0.006903886795043945+0.02243828773498535+0.008210897445678711+0.011810064315795898+0.008182764053344727+0.018049001693725586+0.005044698715209961+0.014535665512084961+0.007874727249145508+0.012639284133911133+0.0075380802154541016+0.017458200454711914+0.007958173751831055+0.02453923225402832+0.011034727096557617+0.012504339218139648+0.007471799850463867+0.02142333984375+0.014215230941772461+0.012639284133911133+0.004713535308837891+0.014199495315551758+0.01849842071533203+0.013310909271240234+0.009598016738891602+0.01665210723876953+0.01861405372619629+0.020431041717529297+0.01483917236328125+0.018316268920898438+0.013285160064697266+0.0263364315032959+0.007675647735595703+0.012762308120727539+0.012038707733154297+0.021584510803222656+0.007609844207763672+0.02272200584411621+0.008038759231567383+0.017429113388061523+0.014098167419433594+0.01855015754699707+0.007455348968505859+0.014028787612915039+0.014645099639892578+0.01000070571899414+0.004655122756958008+0.005469322204589844+0.007612943649291992+0.003796815872192383+0.006846904754638672+0.003854990005493164+0.005514621734619141+0.00855708122253418+0.0056917667388916016+0.00650334358215332+0.006484031677246094+0.013207197189331055+0.007960796356201172+0.005400657653808594+0.004982709884643555+0.01646566390991211+0.006039142608642578+0.006184577941894531+0.007645130157470703+0.018392324447631836+0.005520343780517578+0.003987789154052734+0.01177358627319336+0.01865530014038086+0.012397050857543945+0.005990028381347656+0.009705543518066406+0.016113758087158203+0.013654947280883789+0.00538325309753418+0.023595094680786133+0.018833160400390625+0.007764577865600586+0.0067501068115234375+0.026407480239868164+0.00870823860168457+0.005847930908203125+0.012167692184448242+0.02609705924987793+0.0069539546966552734+0.00736236572265625+0.020289182662963867+0.025823354721069336+0.005423784255981445+0.017802715301513672+0.0246884822845459+0.025842666625976562+0.007502317428588867+0.015811681747436523+0.028374195098876953+0.021078109741210938+0.005394697189331055+0.009296655654907227+0.016928434371948242+0.005135774612426758+0.0072515010833740234+0.011950254440307617+0.017878293991088867+0.01879596710205078+0.008541345596313477
a

11.101118087768555

In [9]:
feasibility, stats = get_stats(payload_wilson_initial,driver_runs)
stats["vmt"]/stats["pmt"], stats["serviced"]/len(payload_wilson_initial["requests"]), np.mean(stats["wait_time"]), np.mean(stats["detour"])

(1.4361121793761844, 0.9329268292682927, 249.27450980392138, 582.350980392157)

In [13]:
with open('../output_wilson/random_weekeday_2.pkl', 'rb') as file:
    rtv_result = pickle.load(file)

feasibility, stats = get_stats(payload_wilson_initial,rtv_result)
stats["vmt"]/stats["pmt"], stats["serviced"]/len(payload_wilson_initial["requests"]), np.mean(stats["wait_time"]), np.mean(stats["detour"])

(1.4361121793761866, 0.9329268292682927, 249.27450980392138, 582.3509803921569)

In [4]:
driver_run = driver_runs[0]
req_2 = payload["requests"][1]
req_4 = payload["requests"][3]
cost, new_driver_run = online_rtv_solver.insert_request_to_driver_run(driver_run, req_2)
feasibility, stats = get_stats(payload_wilson_initial,[new_driver_run])


In [5]:
cost, new_driver_run1 = online_rtv_solver.insert_request_to_driver_run(new_driver_run, req_4)
feasibility, stats = get_stats(payload_wilson_initial,[new_driver_run1])

In [23]:
new_driver_run1

{'state': {'run_id': 0,
  'start_time': 18000,
  'end_time': 72000,
  'am_capacity': 8,
  'wc_capacity': 3,
  'locations_already_serviced': 0,
  'location_dt_seconds': 20043,
  'loc': {'lat': 35.723017652422435, 'lon': -77.90871990823223},
  'total_locations': 4},
 'manifest': [{'run_id': 0,
   'booking_id': 2,
   'order': 1,
   'action': 'pickup',
   'loc': {'lon': -77.943908691, 'lat': 35.709342957, 'node_id': 3},
   'scheduled_time': 20337.6,
   'am': 1,
   'wc': 0,
   'time_window_start': 20161,
   'time_window_end': 21961},
  {'run_id': 0,
   'booking_id': 4,
   'order': 3,
   'action': 'pickup',
   'loc': {'lon': -77.900054932, 'lat': 35.709503174, 'node_id': 0},
   'scheduled_time': 21904.6,
   'am': 1,
   'wc': 0,
   'time_window_start': 20641,
   'time_window_end': 22441},
  {'run_id': 0,
   'booking_id': 2,
   'order': 2,
   'action': 'dropoff',
   'loc': {'lon': -77.996498108, 'lat': 35.733001709, 'node_id': 4},
   'scheduled_time': 21041.1,
   'am': 1,
   'wc': 0,
   'time_

In [ ]:
depot = Node(payload_wilson_initial["depot"]["pt"]["lat"],payload_wilson_initial["depot"]["pt"]["lon"])
req_2o = Node(req_2["pickup_pt"]["lat"],req_2["pickup_pt"]["lon"])
req_2d = Node(req_2["dropoff_pt"]["lat"],req_2["dropoff_pt"]["lon"])
req_4o = Node(req_4["pickup_pt"]["lat"],req_4["pickup_pt"]["lon"])
req_4d = Node(req_4["dropoff_pt"]["lat"],req_4["dropoff_pt"]["lon"])
time_at_depot = driver_run["state"]["location_dt_seconds"]
time_at_req_2o = time_at_depot + NetworkHandler.travel_time(depot,req_2o)
time_at_req_4o = time_at_req_2o + NetworkHandler.travel_time(req_2o,req_4o) + 180
time_at_req_4o + NetworkHandler.travel_time(req_4o,req_4d)

20849.199999999997

In [30]:
import copy
from handlers.payload_parser import PayloadParser

request = req_4
NetworkHandler.init(True, "http://127.0.0.1:50000/")
driver_run_c = copy.deepcopy(new_driver_run)

pickup_stop = {'run_id': None, 'booking_id': request['booking_id'], 'order': -1, 'action': "pickup", 
    "loc": request["pickup_pt"], 'scheduled_time': -1, 
    'am': request["am"], 'wc': request["wc"], 'time_window_start': request['pickup_time_window_start'],
    'time_window_end': request['pickup_time_window_end']}
dropoff_stop = {'run_id': None, 'booking_id': request['booking_id'], 'order': -1, 'action': "dropoff",
    "loc": request["dropoff_pt"], 'scheduled_time': -1, 
    'am': request["am"], 'wc': request["wc"], 'time_window_start': request['dropoff_time_window_start'],
    'time_window_end': request['dropoff_time_window_end']}

node_id = NetworkHandler.get_next_node_id(pickup_stop["loc"]["lat"],pickup_stop["loc"]["lon"])
pickup_stop["loc"]["node_id"] = node_id
node_id = NetworkHandler.get_next_node_id(dropoff_stop["loc"]["lat"],dropoff_stop["loc"]["lon"])
dropoff_stop["loc"]["node_id"] = node_id

load = 0
state = driver_run_c[PayloadParser.DRIVER_STATE]
pickup_stop["run_id"] = state[PayloadParser.DRIVER_STATE_RUN_ID]
dropoff_stop["run_id"] = state[PayloadParser.DRIVER_STATE_RUN_ID]
manifest = driver_run_c[PayloadParser.DRIVER_MANIFEST]
state_loc = state[PayloadParser.DRIVER_STATE_LOC]
node_id = NetworkHandler.get_next_node_id(state_loc["lat"],state_loc["lon"])
state_loc["node_id"] = node_id
start_node = Node(state_loc["lat"],state_loc["lon"],id=node_id)
start_time = state[PayloadParser.DRIVER_STATE_DT_SEC]
completed_stops = []
remaining_stops = []
for stop in manifest:
    if stop["order"] <= state[PayloadParser.DRIVER_STATE_LOC_SERV]:
        if stop["action"] == "pickup":
            load += stop["am"]
        else:
            load -= stop["am"]
        completed_stops.append(stop)
    else:
        remaining_stops.append(stop)
        node_id = NetworkHandler.get_next_node_id(stop["loc"]["lat"],stop["loc"]["lon"])
        stop["loc"]["node_id"] = node_id

NetworkHandler.initialize_travel_time_matrix()

prev_cost = 0
current_node = start_node
for stop in remaining_stops:
    next_node = Node(stop["loc"]["lat"],stop["loc"]["lon"],id=stop["loc"]["node_id"])
    prev_cost += NetworkHandler.travel_time(current_node,next_node)
    current_node = next_node

best_cost = float("inf")
best_insertion = None
for i in range(len(remaining_stops)+1):
    for j in range(i+1,len(remaining_stops)+2):
        new_manifest = copy.deepcopy(remaining_stops[:i] + [pickup_stop] + remaining_stops[i:j] + [dropoff_stop] + remaining_stops[j:])
        # driver_run_c = copy.deepcopy(new_driver_run)
        current_time = start_time
        current_node = start_node
        current_load = load
        cost = 0
        order = state[PayloadParser.DRIVER_STATE_LOC_SERV]
        for stop in new_manifest:
            next_node = Node(stop["loc"]["lat"],stop["loc"]["lon"],id=stop["loc"]["node_id"])
            travel_time = NetworkHandler.travel_time(current_node,next_node)
            cost += travel_time
            current_node = next_node
            current_time += travel_time
            if current_time < stop["time_window_start"]:
                current_time = stop["time_window_start"]
            stop["scheduled_time"] = current_time
            if current_time > stop["time_window_end"]:
                cost = float("inf")
                break
            if stop["action"] == "pickup":
                current_load += stop["am"]
                current_time += 180
            else:
                current_load -= stop["am"]
                current_time += 60
            if current_load > state["am_capacity"]:
                cost = float("inf")
                break
            order += 1
            stop["order"] = order
        print(i,j,cost)
        if cost < best_cost:
            best_cost = cost
            best_insertion = new_manifest

best_cost, best_insertion

0 1 1364.0
0 2 1491.1999999999998
0 3 1491.1999999999998
1 2 1865.9
1 3 1865.9
2 3 2244.7


(1364.0,
 [{'run_id': 0,
   'booking_id': 4,
   'order': 1,
   'action': 'pickup',
   'loc': {'lon': -77.900054932, 'lat': 35.709503174, 'node_id': 0},
   'scheduled_time': 20641,
   'am': 1,
   'wc': 0,
   'time_window_start': 20641,
   'time_window_end': 22441},
  {'run_id': 0,
   'booking_id': 2,
   'order': 2,
   'action': 'pickup',
   'loc': {'lon': -77.943908691, 'lat': 35.709342957, 'node_id': 3},
   'scheduled_time': 21162.7,
   'am': 1,
   'wc': 0,
   'time_window_start': 20161,
   'time_window_end': 21961},
  {'run_id': 0,
   'booking_id': 4,
   'order': 3,
   'action': 'dropoff',
   'loc': {'lon': -77.967590332, 'lat': 35.746833801, 'node_id': 1},
   'scheduled_time': 21777.8,
   'am': 1,
   'wc': 0,
   'time_window_start': 21264.1,
   'time_window_end': 23064.1},
  {'run_id': 0,
   'booking_id': 2,
   'order': 4,
   'action': 'dropoff',
   'loc': {'lon': -77.996498108, 'lat': 35.733001709, 'node_id': 4},
   'scheduled_time': 22237.6,
   'am': 1,
   'wc': 0,
   'time_window_

In [29]:
new_manifest = remaining_stops[:0] + [pickup_stop] + remaining_stops[0:1] + [dropoff_stop] + remaining_stops[1:]
new_manifest

[{'run_id': 0,
  'booking_id': 4,
  'order': 3,
  'action': 'pickup',
  'loc': {'lon': -77.900054932, 'lat': 35.709503174, 'node_id': 0},
  'scheduled_time': 21904.6,
  'am': 1,
  'wc': 0,
  'time_window_start': 20641,
  'time_window_end': 22441},
 {'run_id': 0,
  'booking_id': 2,
  'order': 1,
  'action': 'pickup',
  'loc': {'lon': -77.943908691, 'lat': 35.709342957, 'node_id': 3},
  'scheduled_time': 20337.6,
  'am': 1,
  'wc': 0,
  'time_window_start': 20161,
  'time_window_end': 21961},
 {'run_id': 0,
  'booking_id': 4,
  'order': 4,
  'action': 'dropoff',
  'loc': {'lon': -77.967590332, 'lat': 35.746833801, 'node_id': 1},
  'scheduled_time': 22707.699999999997,
  'am': 1,
  'wc': 0,
  'time_window_start': 21264.1,
  'time_window_end': 23064.1},
 {'run_id': 0,
  'booking_id': 2,
  'order': 2,
  'action': 'dropoff',
  'loc': {'lon': -77.996498108, 'lat': 35.733001709, 'node_id': 4},
  'scheduled_time': 21041.1,
  'am': 1,
  'wc': 0,
  'time_window_start': 20684.5,
  'time_window_end

In [9]:
new_payload_c = {
    "depot": payload_wilson_initial["depot"],
    "requests": [new_payload["requests"][1],new_payload["requests"][3]],
    "driver_runs": new_payload["driver_runs"][:1]}
new_driver_runs, added_cost = online_rtv_solver.solve_rtv_fast(current_time,new_payload_c)
feasibility, stats = get_stats(payload_wilson_initial,new_driver_runs)
new_driver_runs, added_cost

Error: Scheduled time is impossible  1844.5999999999985
Current time:  22885.699999999997
Scheduled time:  21041.1
{'run_id': 0, 'booking_id': 2, 'order': 2, 'action': 'dropoff', 'loc': {'lon': -77.996498108, 'lat': 35.733001709, 'node_id': 4}, 'scheduled_time': 21041.1, 'am': 1, 'wc': 0, 'time_window_start': 20684.5, 'time_window_end': 22484.5}
Error: Scheduled time is impossible  676.5999999999985
Current time:  23384.299999999996
Scheduled time:  22707.699999999997
{'run_id': 0, 'booking_id': 4, 'order': 4, 'action': 'dropoff', 'loc': {'lon': -77.967590332, 'lat': 35.746833801, 'node_id': 1}, 'scheduled_time': 22707.699999999997, 'am': 1, 'wc': 0, 'time_window_start': 21264.1, 'time_window_end': 23064.1}


([{'state': {'run_id': 0,
    'start_time': 18000,
    'end_time': 72000,
    'am_capacity': 8,
    'wc_capacity': 3,
    'locations_already_serviced': 0,
    'location_dt_seconds': 20043,
    'loc': {'lat': 35.723017652422435, 'lon': -77.90871990823223},
    'total_locations': 4},
   'manifest': [{'run_id': 0,
     'booking_id': 2,
     'order': 1,
     'action': 'pickup',
     'loc': {'lon': -77.943908691, 'lat': 35.709342957, 'node_id': 3},
     'scheduled_time': 20337.6,
     'am': 1,
     'wc': 0,
     'time_window_start': 20161,
     'time_window_end': 21961},
    {'run_id': 0,
     'booking_id': 4,
     'order': 3,
     'action': 'pickup',
     'loc': {'lon': -77.900054932, 'lat': 35.709503174, 'node_id': 0},
     'scheduled_time': 21904.6,
     'am': 1,
     'wc': 0,
     'time_window_start': 20641,
     'time_window_end': 22441},
    {'run_id': 0,
     'booking_id': 2,
     'order': 2,
     'action': 'dropoff',
     'loc': {'lon': -77.996498108, 'lat': 35.733001709, 'node_id':

In [12]:
depot = Node(35.72301765242243,-77.90871990823223)
r1o = Node(35.709342957,-77.943908691)
r1d = Node(35.72301765242243,-77.90871990823223)
r2o = Node(35.72301765242243,-77.90871990823223)
r2d = Node(35.709342957,-77.943908691)
time_at_r1o = 18000+NetworkHandler.travel_time(depot,r1o)
time_at_r2o = time_at_r1o+ NetworkHandler.travel_time(r1o,r2o)
time_at_r1o,time_at_r2o
# +NetworkHandler.travel_time(depot,r10)+

(18294.6, 18587.1)

In [8]:
feasibility, stats = get_stats(payload_wilson_initial,new_driver_runs)

In [17]:
NetworkHandler.init(True, online_rtv_solver.server_url)
20043+NetworkHandler.travel_time(Node(35.723017652422435, -77.90871990823223),Node(35.709342957,-77.943908691))+180+NetworkHandler.travel_time(Node(35.709342957,-77.943908691),Node(35.733001709, -77.996498108))+60+NetworkHandler.travel_time(Node(35.733001709, -77.996498108),Node(35.709503174, -77.900054932))+180+NetworkHandler.travel_time(Node(35.709503174, -77.900054932),Node(35.746833801, -77.967590332))

22707.699999999997

In [27]:
a = []
a[1:]

[]

In [9]:
# new_driver_runs, added_cost = online_rtv_solver.solve_rtv_fast(current_time,new_payload)
# new_driver_runs, added_cost

request = new_payload["requests"][3]
print(new_driver_runs[1])

insert_request_to_driver_run(new_driver_runs[1], request)

{'state': {'run_id': 1, 'start_time': 18000, 'end_time': 72000, 'am_capacity': 8, 'wc_capacity': 3, 'locations_already_serviced': 0, 'location_dt_seconds': 20043, 'loc': {'lat': 35.723017652422435, 'lon': -77.90871990823223}, 'total_locations': 2}, 'manifest': [{'run_id': 1, 'booking_id': 2, 'order': 1, 'action': 'pickup', 'loc': {'lon': -77.943908691, 'lat': 35.709342957, 'node_id': 0}, 'scheduled_time': 20337.6, 'am': 1, 'wc': 0, 'time_window_start': 20161, 'time_window_end': 21961}, {'run_id': 1, 'booking_id': 2, 'order': 2, 'action': 'dropoff', 'loc': {'lon': -77.996498108, 'lat': 35.733001709, 'node_id': 1}, 'scheduled_time': 21041.1, 'am': 1, 'wc': 0, 'time_window_start': 20684.5, 'time_window_end': 22484.5}]}
[{'run_id': 1, 'booking_id': 4, 'order': 1, 'action': 'pickup', 'loc': {'lon': -77.900054932, 'lat': 35.709503174, 'node_id': 0}, 'scheduled_time': 20230.4, 'am': 1, 'wc': 0, 'time_window_start': 20641, 'time_window_end': 22441}, {'run_id': 1, 'booking_id': 2, 'order': 2, '

(1047.8000000000002,
 {'state': {'run_id': 1,
   'start_time': 18000,
   'end_time': 72000,
   'am_capacity': 8,
   'wc_capacity': 3,
   'locations_already_serviced': 0,
   'location_dt_seconds': 20043,
   'loc': {'lat': 35.723017652422435, 'lon': -77.90871990823223},
   'total_locations': 4},
  'manifest': [{'run_id': 1,
    'booking_id': 2,
    'order': 1,
    'action': 'pickup',
    'loc': {'lon': -77.943908691, 'lat': 35.709342957, 'node_id': 3},
    'scheduled_time': 20337.6,
    'am': 1,
    'wc': 0,
    'time_window_start': 20161,
    'time_window_end': 21961},
   {'run_id': 1,
    'booking_id': 4,
    'order': 3,
    'action': 'pickup',
    'loc': {'lon': -77.900054932, 'lat': 35.709503174, 'node_id': 0},
    'scheduled_time': 21904.6,
    'am': 1,
    'wc': 0,
    'time_window_start': 20641,
    'time_window_end': 22441},
   {'run_id': 1,
    'booking_id': 2,
    'order': 2,
    'action': 'dropoff',
    'loc': {'lon': -77.996498108, 'lat': 35.733001709, 'node_id': 4},
    'sch

In [6]:
import copy
from handlers.payload_parser import PayloadParser
# new_driver_runs, added_cost = online_rtv_solver.solve_rtv_fast(current_time,new_payload)
# new_driver_runs, added_cost
def insert_request_to_driver_run(driver_run, request):
    NetworkHandler.init(True, online_rtv_solver.server_url)
    driver_run_c = copy.deepcopy(driver_run)

    pickup_stop = {'run_id': None, 'booking_id': request['booking_id'], 'order': -1, 'action': "pickup", 
        "loc": request["pickup_pt"], 'scheduled_time': -1, 
        'am': request["am"], 'wc': request["wc"], 'time_window_start': request['pickup_time_window_start'],
        'time_window_end': request['pickup_time_window_end']}
    dropoff_stop = {'run_id': None, 'booking_id': request['booking_id'], 'order': -1, 'action': "dropoff",
        "loc": request["dropoff_pt"], 'scheduled_time': -1, 
        'am': request["am"], 'wc': request["wc"], 'time_window_start': request['dropoff_time_window_start'],
        'time_window_end': request['dropoff_time_window_end']}
    
    node_id = NetworkHandler.get_next_node_id(pickup_stop["loc"]["lat"],pickup_stop["loc"]["lon"])
    pickup_stop["loc"]["node_id"] = node_id
    node_id = NetworkHandler.get_next_node_id(dropoff_stop["loc"]["lat"],dropoff_stop["loc"]["lon"])
    dropoff_stop["loc"]["node_id"] = node_id

    load = 0
    state = driver_run_c[PayloadParser.DRIVER_STATE]
    pickup_stop["run_id"] = state[PayloadParser.DRIVER_STATE_RUN_ID]
    dropoff_stop["run_id"] = state[PayloadParser.DRIVER_STATE_RUN_ID]
    manifest = driver_run_c[PayloadParser.DRIVER_MANIFEST]
    state_loc = state[PayloadParser.DRIVER_STATE_LOC]
    node_id = NetworkHandler.get_next_node_id(state_loc["lat"],state_loc["lon"])
    state_loc["node_id"] = node_id
    start_node = Node(state_loc["lat"],state_loc["lon"],id=node_id)
    start_time = state[PayloadParser.DRIVER_STATE_DT_SEC]
    completed_stops = []
    remaining_stops = []
    for stop in manifest:
        if stop["order"] <= state[PayloadParser.DRIVER_STATE_LOC_SERV]:
            if stop["action"] == "pickup":
                load += stop["am"]
            else:
                load -= stop["am"]
            completed_stops.append(stop)
        else:
            remaining_stops.append(stop)
            node_id = NetworkHandler.get_next_node_id(stop["loc"]["lat"],stop["loc"]["lon"])
            stop["loc"]["node_id"] = node_id
    
    NetworkHandler.initialize_travel_time_matrix()

    prev_cost = 0
    current_node = start_node
    for stop in remaining_stops:
        next_node = Node(stop["loc"]["lat"],stop["loc"]["lon"],id=stop["loc"]["node_id"])
        prev_cost += NetworkHandler.travel_time(current_node,next_node)
        current_node = next_node
    
    best_cost = float("inf")
    best_insertion = None
    for i in range(len(remaining_stops)+1):
        for j in range(i+1,len(remaining_stops)+2):
            new_manifest = remaining_stops[:i] + [pickup_stop] + remaining_stops[i:j] + [dropoff_stop] + remaining_stops[j:]
            # print(new_manifest)
            current_time = start_time
            current_node = start_node
            current_load = load
            cost = 0
            order = state[PayloadParser.DRIVER_STATE_LOC_SERV]
            for stop in new_manifest:
                next_node = Node(stop["loc"]["lat"],stop["loc"]["lon"],id=stop["loc"]["node_id"])
                travel_time = NetworkHandler.travel_time(current_node,next_node)
                cost += travel_time
                current_node = next_node
                current_time += travel_time
                if current_time < stop["time_window_start"]:
                    cost = stop["time_window_start"]
                stop["scheduled_time"] = current_time
                if current_time > stop["time_window_end"]:
                    cost = float("inf")
                    break
                if stop["action"] == "pickup":
                    current_load += stop["am"]
                    current_time += 180
                else:
                    current_load -= stop["am"]
                    current_time += 60
                if current_load > state["am_capacity"]:
                    cost = float("inf")
                    break
                order += 1
                stop["order"] = order
            if cost < best_cost:
                best_cost = cost
                best_insertion = new_manifest
                print(new_manifest)

    if best_insertion is None:
        return -1,None

    new_driver_run = copy.deepcopy(driver_run)
    new_driver_run[PayloadParser.DRIVER_MANIFEST] = completed_stops + best_insertion
    new_driver_run[PayloadParser.DRIVER_STATE][PayloadParser.DRIVER_STATE_T_LOCS] = len(new_driver_run[PayloadParser.DRIVER_MANIFEST])
    return best_cost-prev_cost,new_driver_run

In [6]:
rtv_stats = get_stats(payload_wilson_initial,driver_runs)

  # {'run_id': 0,
  #  'booking_id': 70.0,
  #  'order': 33,
  #  'action': 'pickup',
  #  'loc': {'lat': 35.717411041, 'lon': -77.897705078, 'node_id': 9.0},
  #  'scheduled_time': 38428.0,
  #  'am': 1.0,
  #  'wc': 0.0,
  #  'time_window_start': 38428.0,
  #  'time_window_end': 40228.0},

In [9]:
rtv_stats[1]

{'vmt': 78793.29999999996,
 'pmt': 54865.700000000026,
 'serviced': 153,
 'wait_time': [0.0,
  1591.2999999999993,
  0.0,
  0.0,
  652.4000000000015,
  653.9000000000015,
  1175.0000000000036,
  477.20000000000437,
  0.0,
  0.0,
  21.599999999998545,
  8.19999999999709,
  166.29999999999563,
  1086.8999999999942,
  539.5999999999913,
  0.0,
  825.0999999999985,
  29.30000000000291,
  434.8000000000029,
  0.0,
  0.0,
  445.1999999999971,
  169.89999999999418,
  1265.0999999999913,
  486.09999999999127,
  70.90000000000146,
  276.5,
  0.0,
  122.0,
  1015.3000000000029,
  0.0,
  0.0,
  301.3000000000029,
  422.90000000000146,
  580.1999999999971,
  2.3000000000029104,
  0.0,
  736.1999999999971,
  1096.199999999997,
  176.59999999999854,
  0.0,
  58.79999999999927,
  0.0,
  0.0,
  689.8999999999978,
  0.0,
  94.90000000000146,
  75.40000000000146,
  0.0,
  168.09999999999854,
  96.89999999999418,
  669.0999999999913,
  767.1999999999898,
  1055.2999999999884,
  0.0,
  0.0,
  0.0,
  369.0

In [19]:
38428+180+NetworkHandler.travel_time(Node(35.717411041, -77.897705078), Node(35.723246, -77.917038)),38769.1

(38767.1, 38769.1)

In [2]:
from handlers.request_handler import RequestHandler
from handlers.network_handler import NetworkHandler
from handlers.vehicle_handler import VehicleHandler
from handlers.trip_handler import TripHandler
from handlers.payload_parser import PayloadParser
import copy

MAX_CARDINALITY = 4
MAX_THREAD_CNT = 64
REBALANCING = False
RH_FACTOR = 1
DWELL_PICKUP = 180
DWELL_ALIGHT = 60
SHAREABLE_COST_FACTOR=1
RTV_TIMEOUT=3000
LARGEST_TSP = 10
ILP_SOLVER_TIMEOUT = 120 # seconds
PENALTY = 1000000 # penalty for not serving a trip

NetworkHandler.init(True, "http://127.0.0.1:50000/")

payload_object = PayloadParser.get_payload_object(new_payload)
request_handler = RequestHandler(payload_object.requests, DWELL_PICKUP, DWELL_ALIGHT)
temp_batch = request_handler.get_all_requests()
batch = []
active_requests = {}
boarded_requests = {}
for req in temp_batch:
    req_id = req.id
    if req_id in payload_object.boarded_requests:
        boarded_requests[req_id] = req
    else:
        if req_id in payload_object.active_requests:
            active_requests[req_id] = req
        batch.append(req)

iteration = 0
boarded_trips = TripHandler.create_trip_for_picked_requests(boarded_requests,iteration)

vehicle_handler = VehicleHandler(payload_object.depot, payload_object.driver_runs,None,LARGEST_TSP=LARGEST_TSP)
vehicle_handler.add_manifest_to_vehicles(payload_object.driver_runs,boarded_requests,boarded_trips,DWELL_ALIGHT, DWELL_PICKUP)

NetworkHandler.initialize_travel_time_matrix()
iteration+=1
trip_handler = TripHandler(current_time,vehicle_handler.vehicles,batch, active_requests, iteration, ILP_SOLVER_TIMEOUT,PENALTY,MAX_CARDINALITY,MAX_THREAD_CNT,SHAREABLE_COST_FACTOR,REBALANCING,RTV_TIMEOUT)
for vehicle_id in trip_handler.vehicle_assignment:
    vehicle = vehicle_handler.vehicles[vehicle_id]
    trips = trip_handler.vehicle_assignment[vehicle_id]
    VehicleHandler.add_new_trips(current_time, vehicle, trips, add=True)

# create updated driver runs
updated_driver_runs = []
for driver_run in payload_object.driver_runs:
    state = driver_run[PayloadParser.DRIVER_STATE]
    manifest = driver_run[PayloadParser.DRIVER_MANIFEST]
    current_order = state[PayloadParser.DRIVER_STATE_LOC_SERV]
    new_manifest = manifest[:current_order]
    vehicle = vehicle_handler.vehicles[state[PayloadParser.DRIVER_STATE_RUN_ID]]
    new_manifest.extend(VehicleHandler.get_manifest(vehicle,current_order))
    state[PayloadParser.DRIVER_STATE_T_LOCS] = len(new_manifest)
    new_driver_run = {PayloadParser.DRIVER_STATE:state,PayloadParser.DRIVER_MANIFEST:new_manifest}
    updated_driver_runs.append(new_driver_run)



NameError: name 'new_payload' is not defined

In [66]:
driver_runs[1]["state"], current_time

({'run_id': 1,
  'start_time': 18000,
  'end_time': 72000,
  'am_capacity': 8,
  'wc_capacity': 3,
  'locations_already_serviced': 33,
  'location_dt_seconds': 38769.1,
  'loc': {'lat': 35.723246, 'lon': -77.917038},
  'total_locations': 36},
 38643)

In [69]:
driver_runs[1]["manifest"][-1]

{'run_id': 1,
 'booking_id': 70.0,
 'order': 34,
 'action': 'dropoff',
 'loc': {'lat': 35.727714539, 'lon': -77.922805786, 'node_id': 10.0},
 'scheduled_time': 38853.0,
 'am': 1.0,
 'wc': 0.0,
 'time_window_start': 38673.0,
 'time_window_end': 40473.0}

In [68]:
boarded_requests

{67.0: <structure.request.Request at 0x7fc1db2e9840>,
 70.0: <structure.request.Request at 0x7fc1db2eacb0>}

In [37]:
coordinates = []
for node in NetworkHandler.node_data:
    coordinate = "{0},{1}".format(node["lon"],node["lat"])
    coordinates.append(coordinate)
url="{0}{1}".format("http://127.0.0.1:50000/table/v1/driving/",";".join(coordinates[:100]))
res = NetworkHandler.get_response(url)

In [ ]:
trip_handler.vehicle_assignment[1]

38643

In [47]:
batch[0].id,batch[0].pick_up_time,batch[0].latest_pick_up_time

(66.0, 38091.0, 39891.0)

In [ ]:
from offline_rtv_solver import OfflineRTVSolver
interval = 10*60 # 10 minutes
step_size = 10*60 # 10 minutes
offline_solver = OfflineRTVSolver("http://127.0.0.1:50000/")
offline_driver_runs = offline_solver.solve_rtv(payload,interval,step_size)

NameError: name 'OfflineRTVSolver' is not defined

In [ ]:
from offline_rtv_solver import OfflineRTVSolver
import pickle
import numpy as np
import os



file = open('../inputs/wilson/two_peak_demand_0.pkl', 'rb')
payload_wilson_initial = pickle.load(file)
file.close()

interval = 10*60 # 10 minutes
step_size = 10*60 # 10 minutes
offline_solver = OfflineRTVSolver("http://127.0.0.1:50000/")
offline_driver_runs = offline_solver.solve_rtv(payload_wilson_initial,interval,step_size)

with open('../output_wilson/two_peak_demand_0.pkl', 'wb') as file:
    pickle.dump(driver_runs, file)

In [ ]:
from offline_rtv_solver import OfflineRTVSolver
import pickle
import numpy as np
import os

for filename in os.listdir('../inputs/wilson/'):
    # if "random_weekeday_2.pkl" in filename:
    #     continue
    # filename = "random_weekeday_2.pkl"
    print(filename)
    file = open('../inputs/wilson/'+filename, 'rb')
    payload = pickle.load(file)
    file.close()

    interval = 10*60 # 10 minutes
    step_size = 10*60 # 10 minutes
    offline_solver = OfflineRTVSolver("http://127.0.0.1:50000/")
    offline_driver_runs = offline_solver.solve_rtv(payload,interval,step_size)

    with open('../output_wilson/'+filename, 'wb') as file:
        pickle.dump(offline_driver_runs, file)

random_weekeday_2.pkl
two_peak_demand_6.pkl
random_weekeday_7.pkl
weekday_demand_1.pkl
weekday_demand_4.pkl
weekday_demand_8.pkl
two_peak_demand_5.pkl
weekday_demand_0.pkl
two_peak_demand_9.pkl
weekend_demand.pkl
two_peak_demand_3.pkl
two_peak_demand_4.pkl
random_weekend_0.pkl
weekend_demand_1.pkl
weekday_demand_2.pkl
weekend_demand_2.pkl
two_peak_demand_2.pkl
two_peak_demand_8.pkl
random_weekeday_5.pkl
random_weekeday_6.pkl
weekday_demand_3.pkl
weekday_demand_7.pkl
two_peak_demand_0.pkl
random_weekeday_0.pkl
two_peak_demand_7.pkl
weekday_demand.pkl
weekday_demand_6.pkl
random_weekend_1.pkl
random_weekeday_8.pkl
random_weekeday_1.pkl
random_weekeday_4.pkl
random_weekeday_9.pkl
weekday_demand_9.pkl
two_peak_demand_1.pkl
weekday_demand_5.pkl
random_weekeday_3.pkl


In [ ]:
feasible, stats = get_stats(payload_wilson_initial,offline_driver_runs)
print("Feasible: ",feasible)
print("Total requests: ",len(payload_wilson_initial["requests"]))
print("Serviced: ",stats["serviced"])
print("Service Rate: ",stats["serviced"]/len(payload_wilson_initial["requests"]))
print("VMT: ",stats["vmt"])
print("PMT: ",stats["pmt"])
print("VMT/PMT: ",stats["vmt"]/stats["pmt"])
print("Wait Time: ",np.mean(stats["wait_time"]))
print("Detour: ",np.mean(stats["detour"]))


Feasible:  True
Total requests:  164
Serviced:  153
Service Rate:  0.9329268292682927
VMT:  78793.29999999994
PMT:  54865.700000000026
VMT/PMT:  1.4361121793761842
Wait Time:  249.30065359477106
Detour:  582.3509803921571


In [2]:
from handlers.network_handler import NetworkHandler
from structure.node import Node

def get_stats(payload,manifest):
    feasible = True
    stats = {}
    stats["vmt"] = 0
    stats["pmt"] = 0
    stats["serviced"] = 0
    stats["wait_time"] = []
    stats["detour"] = []

    NetworkHandler.init(True, "http://127.0.0.1:50000/")
    request_stops = {}
    for driver_run in manifest:
        # print(driver_run["state"]["run_id"])
        load = 0
        current_node = Node(payload["depot"]["pt"]["lat"],payload["depot"]["pt"]["lon"])
        current_time = driver_run["state"]["start_time"]
        for stop in driver_run["manifest"]:
            booking_id = stop["booking_id"]
            if booking_id not in request_stops:
                request_stops[booking_id] = {}
            action = stop["action"]
            served_time = stop["scheduled_time"]
            next_node = Node(stop["loc"]["lat"],stop["loc"]["lon"])
            duration = NetworkHandler.travel_time(current_node,next_node)
            stats["vmt"] += duration
            current_time += duration
            if current_time > served_time+0.5:
                feasible = False
                print("Error: Scheduled time is impossible ", current_time-served_time)
                print("Current time: ",current_time)
                print("Scheduled time: ",served_time)
                print(stop)
            if current_time < served_time:
                current_time = served_time
            
            if served_time < stop["time_window_start"]:
                feasible = False
                print("Error: Served before window start")
            if served_time > stop["time_window_end"]:
                feasible = False
                print("Error: Served after window end")
            if action == "pickup":
                load += stop["am"]
                current_time += 180
                if "pick_up" in request_stops[booking_id]:
                    print("Error: Pick up already exists")
                request_stops[booking_id]["pick_up"] = stop
            else:
                current_time += 60
                load -= stop["am"]
                if "drop_off" in request_stops[booking_id]:
                    feasible = False
                    print("Error: Drop off already exists")
                if "pick_up" not in request_stops[booking_id]:
                    feasible = False
                    print("Error: Drop off before pick up")
                request_stops[booking_id]["drop_off"] = stop
                stats["serviced"] += 1
            if load > driver_run["state"]["am_capacity"]:
                feasible = False
                print("Error: Over capacity")
            current_node = next_node

    for served in request_stops:
        if "drop_off" not in request_stops[served]:
            feasible = False
            print("Error: Request not dropped off")
        origin = Node(request_stops[served]["pick_up"]["loc"]["lat"],request_stops[served]["pick_up"]["loc"]["lon"])
        destination = Node(request_stops[served]["drop_off"]["loc"]["lat"],request_stops[served]["drop_off"]["loc"]["lon"])
        travel_time = NetworkHandler.travel_time(origin,destination)
        stats["pmt"] += travel_time
        stats["wait_time"].append(request_stops[served]["pick_up"]["scheduled_time"]-request_stops[served]["pick_up"]["time_window_start"])
        stats["detour"].append(request_stops[served]["drop_off"]["scheduled_time"]-request_stops[served]["pick_up"]["scheduled_time"]-travel_time)
    return feasible, stats

# feasible, stats = get_stats(payload_wilson_initial,driver_runs)

In [9]:
print("Total requests: ",len(payload_wilson_initial["requests"]))
print("Serviced: ",stats["serviced"])
print("Service Rate: ",stats["serviced"]/len(payload_wilson_initial["requests"]))
print("VMT: ",stats["vmt"])
print("PMT: ",stats["pmt"])
print("VMT/PMT: ",stats["vmt"]/stats["pmt"])
print("Wait Time: ",np.mean(stats["wait_time"]))
print("Detour: ",np.mean(stats["detour"]))

Total requests:  164
Serviced:  153
Service Rate:  0.9329268292682927
VMT:  78793.30000000003
PMT:  54865.70000000002
VMT/PMT:  1.436112179376186
Wait Time:  249.27450980392138
Detour:  582.3509803921571


In [10]:
with open('../output_wilson/stats.csv', 'a+') as stats_file:
    stats_file.write("filename,requests,serviced,service_rate,vmt,pmt,vmt/pmt,wait_time,detour\n")
    for filename in os.listdir('../inputs/wilson/'):
        # if "two_peak_demand_0" not in filename:
        #     continue
        # filename = "random_weekeday_2.pkl"
        print(filename)
        file = open('../inputs/wilson/'+filename, 'rb')
        payload = pickle.load(file)
        file.close()

        file = open('../output_wilson/'+filename, 'rb')
        manifest = pickle.load(file)
        file.close()

        feasible, stats = get_stats(payload,manifest)

        print("Total requests: ",len(payload["requests"]))
        print("Serviced: ",stats["serviced"])
        print("Service Rate: ",stats["serviced"]/len(payload["requests"]))
        print("VMT: ",stats["vmt"])
        print("PMT: ",stats["pmt"])
        print("VMT/PMT: ",stats["vmt"]/stats["pmt"])
        print("Wait Time: ",np.mean(stats["wait_time"]))
        print("Detour: ",np.mean(stats["detour"]))
        stats_file.write("{0},{1},{2},{3},{4},{5},{6},{7},{8}\n".format(filename.split(".")[0],len(payload["requests"]),stats["serviced"],stats["serviced"]/len(payload["requests"]),stats["vmt"],stats["pmt"],stats["vmt"]/stats["pmt"],np.mean(stats["wait_time"]),np.mean(stats["detour"])))

random_weekeday_2.pkl
Total requests:  164
Serviced:  153
Service Rate:  0.9329268292682927
VMT:  78793.30000000003
PMT:  54865.69999999999
VMT/PMT:  1.4361121793761866
Wait Time:  249.27450980392138
Detour:  582.3509803921569
two_peak_demand_6.pkl
Total requests:  281
Serviced:  212
Service Rate:  0.7544483985765125
VMT:  102176.60000000002
PMT:  77376.69999999998
VMT/PMT:  1.320508628566481
Wait Time:  384.00330188679294
Detour:  611.9500000000002
random_weekeday_7.pkl
Total requests:  243
Serviced:  208
Service Rate:  0.8559670781893004
VMT:  95813.89999999997
PMT:  78875.40000000004
VMT/PMT:  1.2147500995240585
Wait Time:  373.56346153846215
Detour:  658.0475961538458
weekday_demand_1.pkl
Total requests:  321
Serviced:  246
Service Rate:  0.7663551401869159
VMT:  106343.69999999994
PMT:  88237.80000000005
VMT/PMT:  1.2051943724798202
Wait Time:  389.234959349593
Detour:  658.0390243902436
weekday_demand_4.pkl
Total requests:  321
Serviced:  252
Service Rate:  0.7850467289719626
VMT

In [16]:
current_time = 37098 + 60 + NetworkHandler.travel_time(Node(35.724243164,-77.907318115),Node(35.717411041,-77.897705078))
# current_time = 38428+180 + NetworkHandler.travel_time(Node(35.717411041,-77.897705078),Node(35.720275879,-77.961013794))
current_time

37275

In [18]:
38428 + 180 + NetworkHandler.travel_time(Node(35.717411041,-77.897705078),Node(35.720275879,-77.961013794))

39060.6

In [4]:
from handlers.network_handler import NetworkHandler

In [39]:
from handlers.request_handler import RequestHandler
from handlers.network_handler import NetworkHandler
from handlers.vehicle_handler import VehicleHandler
from handlers.trip_handler import TripHandler
from handlers.payload_parser import PayloadParser
import copy

MAX_CARDINALITY = 4
MAX_THREAD_CNT = 64
REBALANCING = False
RH_FACTOR = 1
DWELL_PICKUP = 180
DWELL_ALIGHT = 60
SHAREABLE_COST_FACTOR=1
RTV_TIMEOUT=3000
LARGEST_TSP = 10
ILP_SOLVER_TIMEOUT = 120 # seconds
PENALTY = 1000000 # penalty for not serving a trip

NetworkHandler.init(True, "http://127.0.0.1:50000/")
payload_object = PayloadParser.get_payload_object(new_payload)
request_handler = RequestHandler(payload_object.requests, DWELL_PICKUP, DWELL_ALIGHT)
temp_batch = request_handler.get_all_requests()
batch = []
active_requests = {}
boarded_requests = {}
for req in temp_batch:
    req_id = req.id
    if req_id in payload_object.boarded_requests:
        boarded_requests[req_id] = req
    else:
        if req_id in payload_object.active_requests:
            active_requests[req_id] = req
        batch.append(req)

iteration = 0
boarded_trips = TripHandler.create_trip_for_picked_requests(boarded_requests,iteration)

vehicle_handler = VehicleHandler(payload_object.depot, payload_object.driver_runs,None,LARGEST_TSP=LARGEST_TSP)
vehicle_handler.add_manifest_to_vehicles(payload_object.driver_runs,boarded_requests,boarded_trips,DWELL_ALIGHT, DWELL_PICKUP)

NetworkHandler.initialize_travel_time_matrix()
iteration+=1
# trip_handler = TripHandler(current_time,vehicle_handler.vehicles,batch, active_requests, iteration, ILP_SOLVER_TIMEOUT,PENALTY,MAX_CARDINALITY,MAX_THREAD_CNT,SHAREABLE_COST_FACTOR,REBALANCING,RTV_TIMEOUT)
# for vehicle_id in trip_handler.vehicle_assignment:
#     vehicle = vehicle_handler.vehicles[vehicle_id]
#     trips = trip_handler.vehicle_assignment[vehicle_id]
#     VehicleHandler.add_new_trips(current_time, vehicle, trips, add=True)


In [40]:
print(boarded_requests[171.0].origin),boarded_trips[3].request_id,print(boarded_trips[3])

{lat: 35.752407074, lon: -77.930862427, id: 11.0}
{ID: 0-171.0, time: 58239.0, origin: {lat: 35.752407074, lon: -77.930862427, id: 11.0}, destination: {lat: 35.772975922, lon: -77.942726135, id: 12.0}}


(None, 171.0, None)

In [41]:
print(vehicle_handler.vehicles[1].stop_sequence[0])

{Trip ID: 0-171.0, node: {lat: 35.772975922, lon: -77.942726135, id: 12.0}, type: 1}


In [42]:
len(NetworkHandler.node_data)

21

In [10]:
# create updated driver runs
driver_run = payload_object.driver_runs[1]
state = driver_run[PayloadParser.DRIVER_STATE]
print(driver_run)
manifest = driver_run[PayloadParser.DRIVER_MANIFEST]
current_order = state[PayloadParser.DRIVER_STATE_LOC_SERV]
new_manifest = manifest[:current_order]
vehicle = vehicle_handler.vehicles[state[PayloadParser.DRIVER_STATE_RUN_ID]]
new_manifest.extend(VehicleHandler.get_manifest(vehicle,current_order))

{'state': {'run_id': 1, 'start_time': 18000, 'end_time': 72000, 'am_capacity': 8, 'wc_capacity': 3, 'locations_already_serviced': 73, 'location_dt_seconds': 58855.1, 'loc': {'lat': 35.752407074, 'lon': -77.930862427, 'node_id': 21.0}, 'total_locations': 74}, 'manifest': [{'run_id': 1, 'booking_id': 4.0, 'order': 1, 'action': 'pickup', 'loc': {'lat': 35.720035553, 'lon': -77.893722534, 'node_id': 7.0}, 'scheduled_time': 19987.1, 'am': 1.0, 'wc': 0.0, 'time_window_start': 19932.0, 'time_window_end': 21732.0}, {'run_id': 1, 'booking_id': 5.0, 'order': 2, 'action': 'pickup', 'loc': {'lat': 35.720035553, 'lon': -77.893722534, 'node_id': 9.0}, 'scheduled_time': 20167.1, 'am': 1.0, 'wc': 0.0, 'time_window_start': 19932.0, 'time_window_end': 21732.0}, {'run_id': 1, 'booking_id': 3.0, 'order': 3, 'action': 'pickup', 'loc': {'lat': 35.740200043, 'lon': -77.91268158, 'node_id': 3.0}, 'scheduled_time': 20602.799999999, 'am': 1.0, 'wc': 0.0, 'time_window_start': 20439, 'time_window_end': 21647.0}, 

IndexError: invalid index

In [ ]:
# last_node, time_at_last_node = VehicleHandler.get_current_location_time(vehicle)
print(last_node), time_at_last_node, NetworkHandler.travel_time(last_node,vehicle_handler.vehicles[1].stop_sequence[0].node)

{lat: 35.752407074, lon: -77.930862427, id: 21.0}


(None, 58855.1, 0.0)

In [43]:
print(vehicle.next_immediate_node),print(vehicle.last_node)

{lat: 35.752407074, lon: -77.930862427, id: 21.0}
{lat: 35.752407074, lon: -77.930862427, id: 21.0}


(None, None)

In [19]:
TYPE_PICK_UP = 0
TYPE_DROP_OFF = 1

manifest = []
last_node, time_at_last_node = VehicleHandler.get_current_location_time(vehicle)
for vehicle_stop in vehicle.stop_sequence:
    trip = vehicle.trips[vehicle_stop.trip_id]
    node = vehicle_stop.node
    action = "dropoff"
    time_window_start = trip.earliest_arrival_time
    time_window_end = trip.latest_arrival_time
    dwell = trip.dwell_alight
    if vehicle_stop.type == TYPE_PICK_UP:
        action = "pickup"
        time_window_start = trip.pick_up_time
        time_window_end = trip.latest_pick_up_time
        dwell = trip.dwell_pickup
    current_order+=1
    print(last_node, node)
    stop_time = time_at_last_node + NetworkHandler.travel_time(last_node,node)
    if stop_time <= time_window_start:
        stop_time = time_window_start
    stop = {'run_id': vehicle.id, 'booking_id': trip.request_id, 'order': current_order, 'action': action, 
                "loc": {'lat': node.lat, 'lon': node.lon, 'node_id': node.id}, 'scheduled_time': stop_time, 
                'am': trip.am_capacity, 'wc': trip.wc_capacity, 'time_window_start': time_window_start, 
                'time_window_end':time_window_end}
    last_node, time_at_last_node = node, stop_time + dwell
    manifest.append(stop)

{lat: 35.722660065, lon: -77.899337769, id: 28.0} {lat: 35.714157104, lon: -77.8828125, id: 20.0}


IndexError: invalid index

In [1]:
vehicle.stop_sequence

NameError: name 'vehicle' is not defined

In [2]:
payload_wilson_initial["requests"]

[{'booking_id': 1.0,
  'pickup_pt': {'lon': -77.90247345, 'lat': 35.707904816},
  'dropoff_pt': {'lon': -77.906433105, 'lat': 35.737380981},
  'pickup_time_window_start': 19822,
  'pickup_time_window_end': 21622,
  'dropoff_time_window_start': 20112.7,
  'dropoff_time_window_end': 21912.7,
  'am': 1,
  'wc': 0},
 {'booking_id': 2.0,
  'pickup_pt': {'lon': -77.924423218, 'lat': 35.75359726},
  'dropoff_pt': {'lon': -77.963066101, 'lat': 35.740951538},
  'pickup_time_window_start': 19827,
  'pickup_time_window_end': 21627,
  'dropoff_time_window_start': 20270.9,
  'dropoff_time_window_end': 22070.9,
  'am': 1,
  'wc': 0},
 {'booking_id': 3.0,
  'pickup_pt': {'lon': -77.946334839, 'lat': 35.725849152},
  'dropoff_pt': {'lon': -77.967590332, 'lat': 35.746833801},
  'pickup_time_window_start': 19828,
  'pickup_time_window_end': 21628,
  'dropoff_time_window_start': 20123.7,
  'dropoff_time_window_end': 21923.7,
  'am': 1,
  'wc': 0},
 {'booking_id': 4.0,
  'pickup_pt': {'lon': -77.909339905

In [4]:
# Initially all the manifests are empty and the vehicles are at depot location
payload_wilson_initial["driver_runs"]

[{'state': {'run_id': 0,
   'start_time': 18000,
   'end_time': 72000,
   'am_capacity': 8,
   'wc_capacity': 3,
   'locations_already_serviced': 0,
   'location_dt_seconds': 0,
   'loc': {'lat': 35.723017652422435, 'lon': -77.90871990823223}},
  'manifest': []},
 {'state': {'run_id': 1,
   'start_time': 18000,
   'end_time': 72000,
   'am_capacity': 8,
   'wc_capacity': 3,
   'locations_already_serviced': 0,
   'location_dt_seconds': 0,
   'loc': {'lat': 35.723017652422435, 'lon': -77.90871990823223}},
  'manifest': []},
 {'state': {'run_id': 2,
   'start_time': 18000,
   'end_time': 72000,
   'am_capacity': 8,
   'wc_capacity': 3,
   'locations_already_serviced': 0,
   'location_dt_seconds': 0,
   'loc': {'lat': 35.723017652422435, 'lon': -77.90871990823223}},
  'manifest': []},
 {'state': {'run_id': 3,
   'start_time': 18000,
   'end_time': 72000,
   'am_capacity': 8,
   'wc_capacity': 3,
   'locations_already_serviced': 0,
   'location_dt_seconds': 0,
   'loc': {'lat': 35.723017652

In [5]:
# creating a payload with few requests
# consider all requests that start before 05:40:00
selected_requests = []
for request in payload_wilson_initial["requests"]:
    if request["pickup_time_window_start"] < 5*3600+40*60:
        selected_requests.append(request)

new_payload = {
    "depot": payload_wilson_initial["depot"],
    "requests": selected_requests,
    "driver_runs": payload_wilson_initial["driver_runs"]}

In [6]:
# 7 requests are selected
selected_requests

[{'booking_id': 1.0,
  'pickup_pt': {'lon': -77.90247345, 'lat': 35.707904816},
  'dropoff_pt': {'lon': -77.906433105, 'lat': 35.737380981},
  'pickup_time_window_start': 19822,
  'pickup_time_window_end': 21622,
  'dropoff_time_window_start': 20112.7,
  'dropoff_time_window_end': 21912.7,
  'am': 1,
  'wc': 0},
 {'booking_id': 2.0,
  'pickup_pt': {'lon': -77.924423218, 'lat': 35.75359726},
  'dropoff_pt': {'lon': -77.963066101, 'lat': 35.740951538},
  'pickup_time_window_start': 19827,
  'pickup_time_window_end': 21627,
  'dropoff_time_window_start': 20270.9,
  'dropoff_time_window_end': 22070.9,
  'am': 1,
  'wc': 0},
 {'booking_id': 3.0,
  'pickup_pt': {'lon': -77.946334839, 'lat': 35.725849152},
  'dropoff_pt': {'lon': -77.967590332, 'lat': 35.746833801},
  'pickup_time_window_start': 19828,
  'pickup_time_window_end': 21628,
  'dropoff_time_window_start': 20123.7,
  'dropoff_time_window_end': 21923.7,
  'am': 1,
  'wc': 0},
 {'booking_id': 4.0,
  'pickup_pt': {'lon': -77.909339905

In [3]:
from online_rtv_solver import OnlineRTVSolver

# Initialize the RTV solver with the URL of the OSRM server
online_rtv_solver = OnlineRTVSolver("http://127.0.0.1:50000/")

In [4]:
# Generating a manifest for the selected requests

current_time = 5*3600+30*60
driver_runs = online_rtv_solver.solve_rtv(current_time,new_payload)

NameError: name 'new_payload' is not defined

In [9]:
driver_runs

[{'state': {'run_id': 0,
   'start_time': 18000,
   'end_time': 72000,
   'am_capacity': 8,
   'wc_capacity': 3,
   'locations_already_serviced': 0,
   'location_dt_seconds': 0,
   'loc': {'lat': 35.723017652422435, 'lon': -77.90871990823223},
   'total_locations': 4},
  'manifest': [{'run_id': 0,
    'booking_id': 5.0,
    'order': 1,
    'action': 'pickup',
    'loc': {'lat': 35.720035553, 'lon': -77.893722534, 'node_id': 9.0},
    'scheduled_time': 20002.0,
    'am': 1.0,
    'wc': 0.0,
    'time_window_start': 20002.0,
    'time_window_end': 21802.0},
   {'run_id': 0,
    'booking_id': 6.0,
    'order': 2,
    'action': 'pickup',
    'loc': {'lat': 35.728252411, 'lon': -77.909576416, 'node_id': 11.0},
    'scheduled_time': 20343.5,
    'am': 1.0,
    'wc': 0.0,
    'time_window_start': 20139.0,
    'time_window_end': 21939.0},
   {'run_id': 0,
    'booking_id': 5.0,
    'order': 3,
    'action': 'dropoff',
    'loc': {'lat': 35.753082275, 'lon': -77.930335999, 'node_id': 10.0},
   

In [10]:
# Simulate to 5:40:00

current_time = 5*3600+40*60+00
new_driver_runs = online_rtv_solver.simulate_manifest(current_time,driver_runs)

In [11]:
new_driver_runs

[{'state': {'run_id': 0,
   'start_time': 18000,
   'end_time': 72000,
   'am_capacity': 8,
   'wc_capacity': 3,
   'locations_already_serviced': 2,
   'location_dt_seconds': 20523.5,
   'loc': {'lat': 35.728252411, 'lon': -77.909576416, 'node_id': 11.0},
   'total_locations': 4},
  'manifest': [{'run_id': 0,
    'booking_id': 5.0,
    'order': 1,
    'action': 'pickup',
    'loc': {'lat': 35.720035553, 'lon': -77.893722534, 'node_id': 9.0},
    'scheduled_time': 20002.0,
    'am': 1.0,
    'wc': 0.0,
    'time_window_start': 20002.0,
    'time_window_end': 21802.0},
   {'run_id': 0,
    'booking_id': 6.0,
    'order': 2,
    'action': 'pickup',
    'loc': {'lat': 35.728252411, 'lon': -77.909576416, 'node_id': 11.0},
    'scheduled_time': 20343.5,
    'am': 1.0,
    'wc': 0.0,
    'time_window_start': 20139.0,
    'time_window_end': 21939.0},
   {'run_id': 0,
    'booking_id': 5.0,
    'order': 3,
    'action': 'dropoff',
    'loc': {'lat': 35.753082275, 'lon': -77.930335999, 'node_id'

In [12]:

# check feasibility of time slots

new_payload = {
    "depot": payload_wilson_initial["depot"],
    "request": {'booking_id': 8.0,
        'pickup_pt': {'lon': -77.920219421, 'lat': 35.731918335},
        'dropoff_pt': {'lon': -77.967590332, 'lat': 35.746833801},
        'time_windows' : [
            {'pickup_time_window_start': 20423, 'pickup_time_window_end': 22223, 'dropoff_time_window_start': 20807.1, 'dropoff_time_window_end': 22607.1,},
            {'pickup_time_window_start': 20423+900, 'pickup_time_window_end': 22223+900, 'dropoff_time_window_start': 20807.1+900, 'dropoff_time_window_end': 22607.1+900,},
            {'pickup_time_window_start': 20423+1800, 'pickup_time_window_end': 22223+1800, 'dropoff_time_window_start': 20807.1+1800, 'dropoff_time_window_end': 22607.1+1800,}
        ],
        'am': 1,
        'wc': 0
    },
    "driver_runs": new_driver_runs
}


feasible_windows = online_rtv_solver.check_feasibility(current_time, new_payload)
feasible_windows

[{'pickup_time_window_start': 20423,
  'pickup_time_window_end': 22223,
  'dropoff_time_window_start': 20807.1,
  'dropoff_time_window_end': 22607.1},
 {'pickup_time_window_start': 21323,
  'pickup_time_window_end': 23123,
  'dropoff_time_window_start': 21707.1,
  'dropoff_time_window_end': 23507.1},
 {'pickup_time_window_start': 22223,
  'pickup_time_window_end': 24023,
  'dropoff_time_window_start': 22607.1,
  'dropoff_time_window_end': 24407.1}]

In [13]:
# creating a new payload with new requests
# consider all requests that start before 05:50:00
selected_requests = []
for request in payload_wilson_initial["requests"]:
    if request["pickup_time_window_start"] < 5*3600+50*60 and request["pickup_time_window_start"] >= 5*3600+40*60:
        selected_requests.append(request)

new_payload = {"depot": payload_wilson_initial["depot"],
    "requests": selected_requests,
    "driver_runs": new_driver_runs}

In [14]:
selected_requests

[{'booking_id': 8.0,
  'pickup_pt': {'lon': -77.920219421, 'lat': 35.731918335},
  'dropoff_pt': {'lon': -77.967590332, 'lat': 35.746833801},
  'pickup_time_window_start': 20423,
  'pickup_time_window_end': 22223,
  'dropoff_time_window_start': 20807.1,
  'dropoff_time_window_end': 22607.1,
  'am': 1,
  'wc': 0},
 {'booking_id': 9.0,
  'pickup_pt': {'lon': -77.944015503, 'lat': 35.749149323},
  'dropoff_pt': {'lon': -77.939537048, 'lat': 35.768268585},
  'pickup_time_window_start': 20463,
  'pickup_time_window_end': 22263,
  'dropoff_time_window_start': 20713.5,
  'dropoff_time_window_end': 22513.5,
  'am': 1,
  'wc': 0},
 {'booking_id': 10.0,
  'pickup_pt': {'lon': -77.900192261, 'lat': 35.721466064},
  'dropoff_pt': {'lon': -77.944717407, 'lat': 35.737010956},
  'pickup_time_window_start': 20567,
  'pickup_time_window_end': 22367,
  'dropoff_time_window_start': 20978.3,
  'dropoff_time_window_end': 22778.3,
  'am': 1,
  'wc': 0},
 {'booking_id': 11.0,
  'pickup_pt': {'lon': -77.88786

In [14]:
driver_runs = online_rtv_solver.solve_rtv(current_time,new_payload)

In [15]:
driver_runs

[{'state': {'run_id': 0,
   'start_time': 18000,
   'end_time': 72000,
   'am_capacity': 8,
   'wc_capacity': 3,
   'locations_already_serviced': 2,
   'location_dt_seconds': 20523,
   'loc': {'lat': 35.728252411, 'lon': -77.909576416, 'node_id': 11},
   'total_locations': 4},
  'manifest': [{'run_id': 0,
    'booking_id': 5.0,
    'order': 1,
    'action': 'pickup',
    'loc': {'lat': 35.720035553, 'lon': -77.893722534, 'node_id': 9},
    'scheduled_time': 20002,
    'am': 1,
    'wc': 0,
    'time_window_start': 20002,
    'time_window_end': 21802},
   {'run_id': 0,
    'booking_id': 6.0,
    'order': 2,
    'action': 'pickup',
    'loc': {'lat': 35.728252411, 'lon': -77.909576416, 'node_id': 11},
    'scheduled_time': 20343,
    'am': 1,
    'wc': 0,
    'time_window_start': 20139,
    'time_window_end': 21939},
   {'run_id': 0,
    'booking_id': 5.0,
    'order': 3,
    'action': 'dropoff',
    'loc': {'lat': 35.753082275, 'lon': -77.930335999, 'node_id': 16},
    'scheduled_time':

In [ ]:
import json

with open('example_manifest.json', 'r') as file:
    example_manifest = json.load(file)


In [40]:
import pickle

with open('example_manifest.pkl', 'wb') as file:
    pickle.dump(example_manifest, file)


In [ ]:
import math
import pickle
from datetime import datetime, time, timedelta
import numpy as np

with open('example_manifest.pkl', 'rb') as file:
    manifest_data = pickle.load(file)

def get_distribution_from_manifest(manifest_data,bin_width=10):

    
    bin_width_seconds = bin_width*60
    earliest_time, latest_time = 24*3600, 0


    for driver_run in manifest_data:
        manifest = driver_run["manifest"]
        for stop in manifest:
            if stop["action"] == "pickup":
                time_in_seconds = stop["scheduled_time"]
                if time_in_seconds < earliest_time:
                    earliest_time = time_in_seconds
                if time_in_seconds > latest_time:
                    latest_time = time_in_seconds

    earliest_time, latest_time = bin_width_seconds*(earliest_time//bin_width_seconds), bin_width_seconds*(math.ceil(latest_time/bin_width_seconds))
    earliest_time, latest_time

    time_bins = np.arange(earliest_time, latest_time+bin_width_seconds, bin_width_seconds)
    distribution = np.zeros(len(time_bins)-1)

    for driver_run in manifest_data:
        manifest = driver_run["manifest"]
        for stop in manifest:
            if stop["action"] == "pickup":
                time_in_seconds = stop["scheduled_time"]
                distribution += (time_bins[:-1] <= time_in_seconds) & (time_in_seconds < time_bins[1:])

    time_bins_str = [(datetime.min+timedelta(seconds=int(t))).time().isoformat() for t in time_bins]


    distribution_renamed = {}
    for i in range(len(time_bins)-1):
        distribution_renamed[(time_bins_str[i],time_bins_str[i+1])] = distribution[i]

    return distribution_renamed


distribution_renamed = get_distribution_from_manifest(manifest_data,bin_width=10)
print(distribution_renamed)

In [4]:
distribution_renamed

{('05:30:00', '05:40:00'): 7.0,
 ('05:40:00', '05:50:00'): 6.0,
 ('05:50:00', '06:00:00'): 1.0}

{('05:30:00', '05:40:00'): 7.0,
 ('05:40:00', '05:50:00'): 6.0,
 ('05:50:00', '06:00:00'): 1.0}